# Eksplorasi Data IDX Stock Summary

Notebook untuk melihat bentuk data mentah (panel) sampai fitur yang masuk ke Transformer.

Jalankan dari root repo (`E:\trading-BEI`) dengan venv aktif. Pastikan `data/processed/panel.parquet` sudah dibuat via `scraper.build_panel`.

## 1. Setup & load panel

Cell pertama pindah ke root repo supaya path `data/...` dan `import src...` bekerja walau notebook dijalankan dari folder `notebooks/`.

In [ ]:
import os, sys
# pindah ke root repo (parent dari notebooks/) bila perlu
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
assert os.path.exists('data/processed/panel.parquet'), 'panel belum ada - jalankan scraper.build_panel dulu'
print('panel found OK')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

PANEL = 'data/processed/panel.parquet'
panel = pd.read_parquet(PANEL)
print('shape:', panel.shape)
print('date range:', panel['date'].min().date(), '->', panel['date'].max().date())
print('n tickers:', panel['ticker'].nunique())
print('n trading days:', panel['date'].nunique())
panel.head()

## 2. Struktur & tipe kolom

In [ ]:
panel.dtypes.to_frame('dtype')

In [ ]:
# Statistik ringkas kolom numerik
panel[['open','high','low','close','volume','value','foreign_buy','foreign_sell']].describe().T

## 3. Coverage: jumlah saham per hari & hari bursa per tahun

In [ ]:
per_day = panel.groupby('date')['ticker'].nunique()
print('rata-rata saham/hari:', round(per_day.mean(),1))
per_year = panel.groupby(panel['date'].dt.year)['date'].nunique()
print('\nhari bursa per tahun:')
print(per_year)

fig, ax = plt.subplots(figsize=(11,3))
per_day.plot(ax=ax)
ax.set_title('Jumlah saham diperdagangkan per hari')
ax.set_ylabel('n saham'); plt.tight_layout(); plt.show()

## 4. Kualitas data: missing & volume nol

In [ ]:
print('null per kolom:')
print(panel.isna().sum())
zero_vol = (panel['volume']==0).mean()*100
print(f'\nbaris volume nol: {zero_vol:.1f}%  (saham tidak likuid / suspend)')
# berapa hari per ticker (deteksi delisting / listing baru)
days_per_ticker = panel.groupby('ticker')['date'].nunique().sort_values()
print('\nticker dengan riwayat terpendek (kemungkinan IPO baru / delisting):')
print(days_per_ticker.head(8))

## 5. Lihat satu saham (contoh: BBCA)

In [ ]:
TICK = 'BBCA'  # ganti sesuai selera, mis. 'BBRI','TLKM','GOTO'
one = panel[panel['ticker']==TICK].sort_values('date')
print(one.shape)
fig, ax = plt.subplots(2,1, figsize=(11,6), sharex=True)
ax[0].plot(one['date'], one['close']); ax[0].set_title(f'{TICK} close'); ax[0].set_ylabel('IDR')
ax[1].bar(one['date'], one['volume']); ax[1].set_title('volume'); 
plt.tight_layout(); plt.show()
one.tail()

## 6. Distribusi return harian

In [ ]:
tmp = panel.sort_values(['ticker','date']).copy()
tmp['ret'] = np.log(tmp['close'] / tmp.groupby('ticker')['close'].shift(1))
r = tmp['ret'].replace([np.inf,-np.inf], np.nan).dropna()
r = r[r.abs() < 0.5]  # buang outlier ekstrem utk plot
print('mean:', round(r.mean(),5), '| std:', round(r.std(),5), '| skew:', round(r.skew(),3))
fig, ax = plt.subplots(figsize=(9,3))
ax.hist(r, bins=120); ax.set_title('Distribusi log-return harian (semua saham)')
plt.tight_layout(); plt.show()

## 7. Fitur yang masuk ke model (preprocess + normalize)

In [ ]:
from src.preprocess import compute_features, normalize, FEATURE_COLUMNS, TARGET_COLUMN
feats = normalize(compute_features(panel, horizon=1))
print('feature rows:', feats.shape)
print('feature cols:', FEATURE_COLUMNS)
feats.head()

In [ ]:
# Cek normalisasi cross-sectional: per hari mean~0, std~1
d = feats['date'].iloc[len(feats)//2]
day = feats[feats['date']==d][FEATURE_COLUMNS]
print('tanggal:', pd.Timestamp(d).date(), '| n saham:', len(day))
pd.DataFrame({'mean':day.mean().round(4), 'std':day.std(ddof=0).round(4)})

In [ ]:
# Korelasi antar fitur
import numpy as np
corr = feats[FEATURE_COLUMNS].corr()
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURE_COLUMNS))); ax.set_xticklabels(FEATURE_COLUMNS, rotation=90)
ax.set_yticks(range(len(FEATURE_COLUMNS))); ax.set_yticklabels(FEATURE_COLUMNS)
fig.colorbar(im); ax.set_title('Korelasi fitur'); plt.tight_layout(); plt.show()

## 8. Bentuk input Transformer (satu window)

In [ ]:
from src.dataset import IDXWindowDataset
ds = IDXWindowDataset(feats, lookback=60)
print('jumlah sample:', len(ds))
x, y, meta = ds[0]
x = np.asarray(x)
print('x shape (lookback, n_features):', x.shape)
print('target fwd_return:', float(y))
print('meta:', meta['ticker'], str(meta['date'].date()))

fig, ax = plt.subplots(figsize=(9,4))
im = ax.imshow(x.T, aspect='auto', cmap='RdBu', vmin=-3, vmax=3)
ax.set_yticks(range(len(FEATURE_COLUMNS))); ax.set_yticklabels(FEATURE_COLUMNS)
ax.set_xlabel('hari dalam window (0=terlama, 59=hari t)')
ax.set_title(f'Satu input window: {meta["ticker"]} @ {meta["date"].date()}')
fig.colorbar(im); plt.tight_layout(); plt.show()

## Ringkasan bentuk data

- **Panel** (long): satu baris = satu (tanggal, saham), kolom OHLC + volume/value + foreign flow.
- **Fitur**: 7 fitur kausal ternormalisasi cross-sectional per hari.
- **Input model**: tensor `(lookback=60, n_features=7)` per (saham, hari), target = return besok.
